# Detecção de anomalias (Z-score)

Identifica sessões com consumo/duração estatisticamente fora do padrão. Diferente
dos modelos 7a/7b (que operam sobre agregados por usuário/mês), este roda por
**sessão individual** é o nível de granularidade que o documento da Sprint 01
pede ("sinaliza sessões com valores impossíveis antes que entrem no cálculo de
fatura").

## Variáveis usadas

- `kwh_delivered` — energia entregue na sessão
- `duration_min` — duração da sessão
- `kwh_por_min` — taxa de carregamento (kwh_delivered / duration_min)

A taxa de carregamento é incluída porque é onde a "pegada" da anomalia sintética
está: no gerador (Etapa 3), sessões anômalas multiplicam o kWh teórico por 2.5x-4x
**mantendo a duração original** ou seja, o desvio aparece como uma taxa de
carregamento fisicamente implausível, não isoladamente em kWh ou duração.

## Agrupamento: por ponto de recarga

Pontos de recarga têm potências diferentes (7.4, 11, 22 kW). Sem agrupar, o
Z-score compararia sessões de potências bem diferentes na mesma distribuição
gerando falsos positivos (sessão normal num ponto forte parece anômala frente à
média global) e falsos negativos (sessão anômala num ponto fraco passa
despercebida). O Z-score é calculado **separadamente para cada ponto**.

## Critério de anomalia

`|Z| > 3` em qualquer uma das 3 variáveis (usamos o **máximo dos 3 Z-scores
absolutos** por sessão) convenção padrão de 3 desvios-padrão.

## Validação

Como o dataset simulado já tem uma coluna `anomaly_flag` (gerada artificialmente
na Etapa 3), este notebook pode comparar o que o Z-score encontra **sem ver essa
coluna durante o cálculo** contra o gabarito real — dando uma medida honesta de
precisão/recall do método, não só "rodou sem erro".


## Setup

In [1]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psycopg2
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

pd.set_option("display.max_columns", None)

In [2]:
DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "evchargeops"),
    "user": os.getenv("POSTGRES_USER", "evchargeops"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
}

MODELS_DIR = Path("output")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

Z_THRESHOLD = 3.0

## Carga dos dados

Traz `anomaly_flag` do banco só para VALIDAR o método depois não é usada no
cálculo do Z-score em si (isso seria vazamento de dado; o Z-score é
não-supervisionado por natureza).

status='concluida' apenas sessões interrompidas têm duração/kWh
menores por desenho (cortadas no meio), o que não é uma anomalia de
medição e poluiria a distribuição de referência de cada ponto.

In [ ]:
def fetch_sessions(conn) -> pd.DataFrame:
    query = """
        SELECT
            s.session_id,
            s.user_id,
            s.point_id,
            s.kwh_delivered,
            s.duration_min,
            s.status,
            s.anomaly_flag
        FROM fct_sessoes s
        WHERE s.status = 'concluida'
    """
    return pd.read_sql(query, conn)

In [4]:
conn = psycopg2.connect(**DB_CONFIG)
try:
    sessions_df = fetch_sessions(conn)
finally:
    conn.close()

sessions_df["kwh_por_min"] = sessions_df["kwh_delivered"] / sessions_df["duration_min"]

print(f"{len(sessions_df)} sessões concluídas carregadas.")
print(f"Anomalias reais (gabarito, não usado no cálculo): {sessions_df['anomaly_flag'].sum()}")
sessions_df.head()

7921 sessões concluídas carregadas.
Anomalias reais (gabarito, não usado no cálculo): 171


/tmp/ipykernel_376277/782689755.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,session_id,user_id,point_id,kwh_delivered,duration_min,status,anomaly_flag,kwh_por_min
0,S000001,U0001,P002,23.412,152,concluida,False,0.154026
1,S000002,U0001,P002,24.589,158,concluida,False,0.155627
2,S000003,U0001,P002,27.896,186,concluida,False,0.149978
3,S000004,U0001,P002,5.632,36,concluida,False,0.156444
4,S000005,U0001,P002,40.168,243,concluida,False,0.165300


## Cálculo do Z-score por ponto de recarga

In [5]:
VARS = ["kwh_delivered", "duration_min", "kwh_por_min"]

def compute_zscores(df: pd.DataFrame, group_col: str, value_cols: list[str]) -> pd.DataFrame:
    """Calcula Z-score de cada variável, separadamente por grupo (ponto de
    recarga). group.transform preserva o índice original -- necessário para
    atribuir de volta ao dataframe sem reordenar linhas."""
    result = df.copy()
    for col in value_cols:
        grouped = df.groupby(group_col)[col]
        mean = grouped.transform("mean")
        std = grouped.transform("std")
        # Evita divisão por zero em pontos com variância nula (não deveria
        # ocorrer com volume real de sessões, mas é uma proteção honesta).
        std_safe = std.replace(0, np.nan)
        result[f"z_{col}"] = (df[col] - mean) / std_safe
    return result

sessions_z = compute_zscores(sessions_df, "point_id", VARS)
sessions_z[["point_id"] + VARS + [f"z_{c}" for c in VARS]].head()

,point_id,kwh_delivered,duration_min,kwh_por_min,z_kwh_delivered,z_duration_min,z_kwh_por_min
0,P002,23.412,152,0.154026,0.549783,0.778742,-0.179503
1,P002,24.589,158,0.155627,0.652398,0.878078,-0.149141
2,P002,27.896,186,0.149978,0.940712,1.341650,-0.256301
3,P002,5.632,36,0.156444,-1.000331,-1.141769,-0.133624
4,P002,40.168,243,0.165300,2.010623,2.285349,0.034398


## Critério de anomalia: máximo dos Z-scores absolutos > {Z_THRESHOLD}

In [6]:
z_cols = [f"z_{c}" for c in VARS]
sessions_z["z_max_abs"] = sessions_z[z_cols].abs().max(axis=1)
sessions_z["anomaly_predicted"] = sessions_z["z_max_abs"] > Z_THRESHOLD

print(f"Sessões sinalizadas como anômalas pelo Z-score: {sessions_z['anomaly_predicted'].sum()} de {len(sessions_z)}")

Sessões sinalizadas como anômalas pelo Z-score: 251 de 7921


## Validação contra o gabarito (anomaly_flag simulado)

In [7]:
y_true = sessions_z["anomaly_flag"]
y_pred = sessions_z["anomaly_predicted"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision: {precision:.3f}  (dentre as sinalizadas, quantas eram realmente anômalas)")
print(f"Recall:    {recall:.3f}  (dentre as realmente anômalas, quantas foram capturadas)")
print(f"F1-score:  {f1:.3f}")

cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(
    cm,
    index=["Real: normal", "Real: anômala"],
    columns=["Previsto: normal", "Previsto: anômala"],
)

Precision: 0.681  (dentre as sinalizadas, quantas eram realmente anômalas)
Recall:    1.000  (dentre as realmente anômalas, quantas foram capturadas)
F1-score:  0.810


,Previsto: normal,Previsto: anômala
Real: normal,7670,80
Real: anômala,0,171


## Interpretação

Se o recall for baixo, o limiar `|Z| > 3` pode estar conservador demais para o
padrão de anomalia sintética gerado (multiplicador de 2.5x-4x, que nem sempre
desvia 3 desvios-padrão inteiros da média do ponto, especialmente em pontos com
alta variância natural de uso). Testamos abaixo um limiar mais permissivo para
comparação, sem alterar o `Z_THRESHOLD` de produção -- é só um diagnóstico.

In [8]:
for alt_threshold in [2.0, 2.5, 3.0]:
    pred = sessions_z["z_max_abs"] > alt_threshold
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1_alt = f1_score(y_true, pred, zero_division=0)
    n_flagged = pred.sum()
    print(f"Threshold={alt_threshold}: {n_flagged} sinalizadas | precision={p:.3f} recall={r:.3f} f1={f1_alt:.3f}")

Threshold=2.0: 527 sinalizadas | precision=0.324 recall=1.000 f1=0.490
Threshold=2.5: 346 sinalizadas | precision=0.494 recall=1.000 f1=0.662
Threshold=3.0: 251 sinalizadas | precision=0.681 recall=1.000 f1=0.810


## Persistência: Salva os parâmetros (médias/desvios por ponto) necessários para aplicar o mesmo critério em sessões novas, sem precisar reprocessar o histórico inteiro.

In [9]:
reference_stats = sessions_df.groupby("point_id")[VARS].agg(["mean", "std"])
reference_stats.columns = ["_".join(col) for col in reference_stats.columns]
reference_stats = reference_stats.reset_index()

anomaly_model = {
    "reference_stats": reference_stats,
    "variables": VARS,
    "group_col": "point_id",
    "z_threshold": Z_THRESHOLD,
}

joblib.dump(anomaly_model, MODELS_DIR / "deteccao_anomalias_zscore.joblib")
sessions_z.to_csv(MODELS_DIR / "deteccao_anomalias_scores.csv", index=False)

print("Artefatos salvos em:", MODELS_DIR.resolve())
print(" -", "deteccao_anomalias_zscore.joblib")
print(" -", "deteccao_anomalias_scores.csv")

Artefatos salvos em: /home/vinivaliati/projects/computer_science_fiap_challenge_2026/models/output
 - deteccao_anomalias_zscore.joblib
 - deteccao_anomalias_scores.csv
